# Dataset Comparison Analysis

This notebook demonstrates the new dataset-agnostic approach using `utils.py`.

**Features:**
- Auto-discovers available datasets
- Loads results from organized subdirectories
- Works with any dataset without code changes

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import our utilities
from utils import (
    discover_datasets,
    load_experiment_results,
    get_dataset_metadata,
    summarize_results,
    quick_analysis
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Discover Available Datasets

Automatically find all HDF5 datasets:

In [ ]:
# Auto-discover datasets
datasets = discover_datasets()

print(f"Found {len(datasets)} datasets:\n")
for name, filename in datasets:
    meta = get_dataset_metadata(filename)
    shape = meta.get('shape', 'unknown')
    print(f"  {name:30s} ({filename:30s}) - {shape}")
    if meta.get('has_categories'):
        print(f"    → Multi-category: {meta['category_counts']}")

## 2. Load Dataset Comparison Results

In [ ]:
# Load all dataset comparison results
results = load_experiment_results("../results/dataset_comparison")

print(f"Loaded results for {len(results)} datasets:\n")
for dataset_name, data in results.items():
    num_impls = len(data.get('implementations', []))
    print(f"  {dataset_name}: {num_impls} implementations tested")

## 3. Create Summary Tables

In [ ]:
# Error rates across datasets
error_summary = summarize_results("../results/dataset_comparison", "mean_error_pct")
print("\nMean Error % by Implementation and Dataset:")
print(error_summary)

# Insert times
insert_summary = summarize_results("../results/dataset_comparison", "insert_time_sec")
print("\nInsert Time (seconds):")
print(insert_summary)

# Memory usage
memory_summary = summarize_results("../results/dataset_comparison", "memory_bytes")
print("\nMemory Usage (bytes):")
print(memory_summary / 1024)  # Show in KB

## 4. Visualize Performance

In [ ]:
# Error rate comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Error rates
error_summary.plot(kind='bar', ax=ax1)
ax1.set_title('Mean Error % by Implementation', fontsize=14)
ax1.set_ylabel('Error %')
ax1.set_xlabel('Implementation')
ax1.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Insert times  
insert_summary.plot(kind='bar', ax=ax2)
ax2.set_title('Insert Time by Implementation', fontsize=14)
ax2.set_ylabel('Time (seconds)')
ax2.set_xlabel('Implementation')
ax2.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Multi-Category Analysis

Load and analyze multi-category experiment results:

In [ ]:
# Load multi-category results
mc_results = load_experiment_results("../results/multicategory")

for exp_name, data in mc_results.items():
    print(f"\n{exp_name}:")
    print("=" * 60)
    
    for impl in data.get('implementations', []):
        print(f"\n{impl['name']}:")
        print(f"  Memory: {impl['memory_bytes'] / 1024:.1f} KB")
        print(f"  Mean Error: {impl.get('mean_error_pct', 0):.2f}%")
        print(f"  Isolation Rate: {impl.get('isolation_rate_pct', 0):.1f}%")
        
        if 'categories' in impl:
            print(f"\n  Category Details:")
            for cat in impl['categories']:
                print(f"    Cat {cat['category_id']}: expected={cat['expected']}, "
                      f"measured={cat['measured']}, error={cat['error_pct']:.2f}%")

## 6. Quick Analysis (One-Liner)

For quick exploration:

In [ ]:
# One-line workflow
datasets, results, summary = quick_analysis("dataset_comparison")

print("\nQuick Summary:")
print(summary)

## 7. Export Results

Save summaries for reports:

In [ ]:
# Export to CSV
error_summary.to_csv('../results/summary_error_rates.csv')
insert_summary.to_csv('../results/summary_insert_times.csv')
memory_summary.to_csv('../results/summary_memory_usage.csv')

print("Exported summary tables to ../results/")